# Week 7: Evaluating the Financial Fraud Detector

Evaluation of the fine-tuned fraud detector (Llama 3.2 1B + LoRA). Uses `financial_fraud_detector.py` from Google Drive.

**Colab:**

Available on Colab: https://colab.research.google.com/drive/13UyhU9ZHWMyEyFO-9RL5jafdxvaGjidR?usp=sharing


In [ ]:
# Run this cell first on Colab
try:
    import google.colab
    from google.colab import drive
    drive.mount("/content/drive")
    import sys
    sys.path.append("/content/drive/MyDrive/week7")
    !pip install -q transformers peft accelerate datasets trl torch scikit-learn matplotlib python-dotenv openai
except ImportError:
    import sys
    sys.path.append(".")  # local

In [ ]:
# Imports
from financial_fraud_detector import (
    load_and_split_dataset,
    transaction_to_text,
    load_finetuned_model,
    predict_finetuned,
    run_full_pipeline,
    LORA_SAVE_PATH as DEFAULT_LORA_PATH,
)

import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

%matplotlib inline

In [ ]:
# Configuration
DRIVE_WEEK7 = "/content/drive/MyDrive/week7"
LORA_SAVE_PATH = f"{DRIVE_WEEK7}/fraud_detector_lora"

# Fallback if Drive not mounted (local)
if not Path(DRIVE_WEEK7).exists():
    LORA_SAVE_PATH = str(Path(DEFAULT_LORA_PATH).resolve())
    for _p in ["./fraud_detector_lora"]:
        if Path(_p).exists() and (Path(_p) / "adapter_config.json").exists():
            LORA_SAVE_PATH = str(Path(_p).resolve())
            break

model_exists = Path(LORA_SAVE_PATH).exists() and (Path(LORA_SAVE_PATH) / "adapter_config.json").exists()
print(f"LoRA path: {LORA_SAVE_PATH}")
print(f"Model exists: {model_exists}")


In [ ]:
# Train or load model
if model_exists:
    load_finetuned_model(path=LORA_SAVE_PATH)
    print("Model loaded.")
else:
    print("Training (~5-10 min on T4)...")
    run_full_pipeline(lora_path=LORA_SAVE_PATH)
    print("Training complete.")


In [ ]:
# Load test data
train_df, val_df, test_df = load_and_split_dataset()
test_data = [
    {"text": transaction_to_text(row), "is_fraud": bool(row["is_fraud"])}
    for _, row in test_df.iterrows()
]
print(f"Test data: {len(test_data)} samples")


In [ ]:
def parse_prediction(response: str) -> int:
    r = response.strip().lower()
    return 1 if "fraudulent" in r else 0

def predict_for_eval(datapoint):
    return predict_finetuned(datapoint["text"])

In [ ]:
# Optional: compare with base model via OpenRouter
# from financial_fraud_detector import predict_base_openrouter
# FraudTester.test(lambda dp: predict_base_openrouter(dp["text"]), test_data)


In [ ]:
# FraudTester - classification evaluation framework
GREEN = "\033[92m"
RED = "\033[91m"
RESET = "\033[0m"
COLOR_MAP = {"red": RED, "green": GREEN}

class FraudTester:
    def __init__(self, predictor, data, title=None, size=None):
        self.predictor = predictor
        self.data = data
        self.title = title or getattr(predictor, "__name__", "Predictor").replace("_", " ").title()
        self.size = size if size is not None else len(data)
        self.guesses = []
        self.truths = []
        self.colors = []

    def run_datapoint(self, i):
        datapoint = self.data[i]
        raw = self.predictor(datapoint)
        guess = parse_prediction(raw)
        truth = int(datapoint["is_fraud"])
        correct = guess == truth
        color = "green" if correct else "red"
        self.guesses.append(guess)
        self.truths.append(truth)
        self.colors.append(color)
        label = "OK" if correct else "FAIL"
        short = datapoint["text"][:40] + "..." if len(datapoint["text"]) > 40 else datapoint["text"]
        print(f"{COLOR_MAP[color]}{i+1}: {label} Guess={guess} Truth={truth} {short}{RESET}")

    def chart(self):
        cm = confusion_matrix(self.truths, self.guesses)
        fig, ax = plt.subplots(figsize=(6, 5))
        im = ax.imshow(cm, cmap="Blues")
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(["Legitimate", "Fraudulent"])
        ax.set_yticklabels(["Legitimate", "Fraudulent"])
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center")
        plt.title(f"Confusion Matrix - {self.title}")
        plt.tight_layout()
        plt.show()

    def report(self):
        acc = accuracy_score(self.truths, self.guesses)
        prec = precision_score(self.truths, self.guesses, zero_division=0)
        rec = recall_score(self.truths, self.guesses, zero_division=0)
        f1 = f1_score(self.truths, self.guesses, zero_division=0)
        print(f"\n{self.title}")
        print(f"  Accuracy:  {acc:.2%}")
        print(f"  Precision: {prec:.2%}")
        print(f"  Recall:   {rec:.2%}")
        print(f"  F1:       {f1:.2%}")
        print("\nClassification Report:")
        print(classification_report(self.truths, self.guesses, target_names=["Legitimate", "Fraudulent"]))
        self.chart()

    def run(self):
        for i in range(self.size):
            self.run_datapoint(i)
        self.report()

    @classmethod
    def test(cls, predictor, data, size=None):
        cls(predictor, data, size=size).run()

In [ ]:
# Run evaluation
FraudTester.test(predict_for_eval, test_data)

In [ ]:
# Optional: compare with base model via OpenRouter
# from financial_fraud_detector import predict_base_openrouter
# def predict_base_for_eval(dp): return predict_base_openrouter(dp["text"])
# FraudTester.test(predict_base_for_eval, test_data)

In [ ]:
# Evaluation complete